In [4]:
import os
import random
import numpy as np
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# =========================
# Paths
# =========================
train_dir = "Training"
test_dir = "Test"

# =========================
# Parameters
# =========================
img_size = (100, 100)
batch_size = 32
epochs = 5   # faster than 30 for local CPU demo

# Limit the number of classes/images for faster training
max_classes = 20
max_train_per_class = 200
max_test_per_class = 60

random.seed(42)

# =========================
# Select a subset of classes
# =========================
all_classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
selected_classes = all_classes[:max_classes]

print("Selected classes:", selected_classes)
print("Number of selected classes:", len(selected_classes))

# =========================
# Build dataframe-like lists manually
# =========================
train_files = []
train_labels = []
test_files = []
test_labels = []

for class_name in selected_classes:
    class_train_path = os.path.join(train_dir, class_name)
    class_test_path = os.path.join(test_dir, class_name)

    train_images = [os.path.join(class_train_path, f) for f in os.listdir(class_train_path)
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    test_images = [os.path.join(class_test_path, f) for f in os.listdir(class_test_path)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    random.shuffle(train_images)
    random.shuffle(test_images)

    train_images = train_images[:max_train_per_class]
    test_images = test_images[:max_test_per_class]

    train_files.extend(train_images)
    train_labels.extend([class_name] * len(train_images))

    test_files.extend(test_images)
    test_labels.extend([class_name] * len(test_images))

print("Training images used:", len(train_files))
print("Testing images used:", len(test_files))

# =========================
# Generators from dataframe
# =========================
import pandas as pd

train_df = pd.DataFrame({
    "filename": train_files,
    "class": train_labels
})

test_df = pd.DataFrame({
    "filename": test_files,
    "class": test_labels
})

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filename",
    y_col="class",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filename",
    y_col="class",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)

# =========================
# CNN Model
# =========================
model = Sequential([
    Input(shape=(100, 100, 3)),
    Conv2D(16, (2, 2), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(32, (2, 2), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (2, 2), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (2, 2), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Dropout(0.3),
    Flatten(),

    Dense(150, activation='relu'),
    Dropout(0.4),

    Dense(num_classes, activation='softmax')
])

# =========================
# Summary
# =========================
model.summary()

# =========================
# Compile
# =========================
model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# Train
# =========================
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=test_generator
)

# =========================
# Evaluate
# =========================
loss, accuracy = model.evaluate(test_generator, verbose=1)
print("Test Accuracy:", accuracy)

# =========================
# Confusion Matrix
# =========================
predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

cm = confusion_matrix(true_classes, predicted_classes)

print("Confusion Matrix:")
print(cm)

Selected classes: ['Almonds 1', 'Apple 10', 'Apple 11', 'Apple 12', 'Apple 13', 'Apple 14', 'Apple 17', 'Apple 18', 'Apple 19', 'Apple 20', 'Apple 21', 'Apple 22', 'Apple 23', 'Apple 5', 'Apple 6', 'Apple 7', 'Apple 8', 'Apple 9', 'Apple Braeburn 1', 'Apple Crimson Snow 1']
Number of selected classes: 20
Training images used: 4000
Testing images used: 1200
Found 4000 validated image filenames belonging to 20 classes.
Found 1200 validated image filenames belonging to 20 classes.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)                    │ (None, 99, 99, 16)          │             208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 49, 49, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 48, 48, 32)          │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 24, 24, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 23, 23, 64)          │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 11, 11, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 10, 10, 64)          │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 5, 5, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 5, 5, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 1600)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 150)                 │         240,150 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 150)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 20)                  │           3,020 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 270,162 (1.03 MB)

 Trainable params: 270,162 (1.03 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 50s 385ms/step - accuracy: 0.4360 - loss: 1.7838 - val_accuracy: 0.8417 - val_loss: 0.5713
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.7628 - loss: 0.6883 - val_accuracy: 0.8567 - val_loss: 0.4166
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 87ms/step - accuracy: 0.8763 - loss: 0.3762 - val_accuracy: 0.9792 - val_loss: 0.0966
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 83ms/step - accuracy: 0.9175 - loss: 0.2306 - val_accuracy: 0.9900 - val_loss: 0.0374
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 85ms/step - accuracy: 0.9402 - loss: 0.1798 - val_accuracy: 0.9992 - val_loss: 0.0233
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.9992 - loss: 0.0233     
Test Accuracy: 0.9991666674613953
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step  
Confusion Matrix:
[[60  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 60  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 60  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0